# Closing Auction — Features and Target

Starts from `data/panel_clean.csv`, the cleaned panel built in `01) data_exploration.ipynb`: one row per symbol-day, last pre-close snapshot joined to the realized close, 10,430 rows, ETPs and dead-book symbol-days already excluded. Builds engineered features, then several candidate target definitions (target definition was deliberately left open in `01)`), then a date-based train/test split.

In [1]:
import numpy as np
import pandas as pd

panel = pd.read_csv("data/panel_clean.csv", parse_dates=["date"])
panel.shape

(10430, 19)

## Base features (last snapshot)

`panel_clean.csv` already has the last pre-close snapshot per symbol-day and the realized close (`close_price`), the raw material the actual targets get built from below. First pass: derive the level features from that single row — imbalance direction, spread, size normalized by ADV, and the near/far and open-anchored moves.

In [2]:
panel["signed_shares"] = np.where(panel["side"] == "BUY", panel["shares"],
                            np.where(panel["side"] == "SELL", -panel["shares"], 0.0))
panel["signed_imbalance_ratio"] = panel["signed_shares"] / (panel["paired_shares"] + panel["shares"])

panel["spread"] = panel["ask"] - panel["bid"]
panel["spread_bps"] = panel["spread"] / panel["ref_price"] * 1e4
panel["urgency"] = panel["signed_imbalance_ratio"] * panel["spread_bps"]

panel["shares_over_adv"] = panel["shares"] / (panel["adv"] * 1000)
panel["paired_shares_over_adv"] = panel["paired_shares"] / (panel["adv"] * 1000)

panel["far_price_available"] = panel["far_price"] != 0
panel["near_far_spread_bps"] = np.where(
    panel["far_price_available"], (panel["near_price"] / panel["far_price"] - 1) * 1e4, np.nan
)  # far_price == 0.0 means no standalone closing-auction-only orders existed for this name -- a
   # real absence (mostly ETFs/ETNs), not bad data; explicit NaN + flag rather than a silent inf
panel["intraday_move_bps"] = (panel["ref_price"] / panel["open"] - 1) * 1e4

panel[["signed_imbalance_ratio", "spread_bps", "urgency", "shares_over_adv", "near_far_spread_bps", "intraday_move_bps"]].describe().T

,count,mean,std,min,25%,50%,75%,max
signed_imbalance_ratio,10430.0,0.001513,0.058714,-0.994444,0.000000,0.000000,0.003489,1.000000
spread_bps,10430.0,8.250664,11.810362,0.156050,2.896032,5.509106,9.895916,330.578512
urgency,10430.0,-0.012864,1.812696,-69.586067,0.000000,0.000000,0.012531,43.585278
shares_over_adv,10430.0,0.002343,0.007111,0.000000,0.000000,0.000000,0.002121,0.240281
near_far_spread_bps,10398.0,0.578463,190.259505,-6813.333333,0.000000,0.000000,0.000000,11039.396479
intraday_move_bps,10430.0,-13.763523,349.106798,-4056.521739,-143.444428,-1.292432,125.283791,4678.030303


## Time-series features (momentum, lags, volatility)

`panel_clean.csv` is one row per symbol-day, so it has no time-series information by itself — momentum, lagged returns, and rolling volatility (the most consistently useful features in the literature) all need multiple pre-close snapshots. Reloading a narrow slice of the raw feed — the last 120 seconds before close (15:58:00 to 16:00:00), restricted to the symbol-days already validated in `01)` — to compute those, rather than expanding the panel to multiple rows per symbol-day. That keeps one feature vector per symbol-day, matching the take-home's framing of a single entry decision, while still capturing the time-series signal the papers found mattered most.

In [3]:
import tarfile

CLOSE = "16:00:00"

def load_all(tar_path):
    frames = []
    with tarfile.open(tar_path) as tf:
        for member in tf.getmembers():
            if not member.name.endswith(".csv.gz"):
                continue
            date_str = member.name.split(".")[0]
            d = pd.read_csv(tf.extractfile(member), compression="gzip")
            d = d.drop(columns=["ts.1"])
            d["date"] = pd.to_datetime(date_str, format="%Y%m%d")
            frames.append(d)
    df = pd.concat(frames, ignore_index=True)
    df["ts"] = pd.to_datetime(df["ts"])
    df["local_clock"] = df["local_time"].str.slice(11, 19)
    return df

raw = load_all("data/202503_imbalance.tar")

valid_keys = panel[["symbol", "date"]]
raw = raw.merge(valid_keys, on=["symbol", "date"], how="inner")
raw = raw[(raw["local_clock"] >= "15:58:00") & (raw["local_clock"] < CLOSE)].copy()
raw["signed_shares"] = np.where(raw["side"] == "BUY", raw["shares"],
                          np.where(raw["side"] == "SELL", -raw["shares"], 0.0))
raw["imbalance_ratio"] = raw["shares"] / (raw["paired_shares"] + raw["shares"])
raw["signed_imbalance_ratio"] = raw["signed_shares"] / (raw["paired_shares"] + raw["shares"])
raw = raw.sort_values("ts")

print(f"raw slice: {raw.shape[0]} rows across {raw[['symbol','date']].drop_duplicates().shape[0]} symbol-days")

raw slice: 1251600 rows across 10430 symbol-days


In [4]:
t0 = panel[["symbol", "date", "ts"]].rename(columns={"ts": "t0_ts"}).copy()
t0["t0_ts"] = pd.to_datetime(t0["t0_ts"])

lag_cols = ["ref_price", "imbalance_ratio", "signed_imbalance_ratio"]

for lag_s in [10, 30, 60]:
    targets = t0.copy()
    targets["ts"] = targets["t0_ts"] - pd.Timedelta(seconds=lag_s)
    targets = targets.sort_values("ts")

    lagged = pd.merge_asof(
        targets, raw[["ts", "symbol", "date"] + lag_cols],
        on="ts", by=["symbol", "date"], direction="backward",
    )
    lagged = lagged.rename(columns={c: f"{c}_lag{lag_s}s" for c in lag_cols})
    panel = panel.merge(lagged[["symbol", "date"] + [f"{c}_lag{lag_s}s" for c in lag_cols]],
                         on=["symbol", "date"], how="left")

    panel[f"imbalance_ratio_chg_{lag_s}s"] = panel["imbalance_ratio"] - panel[f"imbalance_ratio_lag{lag_s}s"]
    panel[f"signed_imbalance_ratio_chg_{lag_s}s"] = (
        panel["signed_imbalance_ratio"] - panel[f"signed_imbalance_ratio_lag{lag_s}s"]
    )
    panel[f"ref_price_ret_{lag_s}s_bps"] = (
        panel["ref_price"] / panel[f"ref_price_lag{lag_s}s"] - 1
    ) * 1e4

print("null rate in the 60s lag columns (should be near 0 -- only symbol-days with <60s of pre-close data would miss):")
print(panel[["ref_price_lag60s", "imbalance_ratio_lag60s"]].isna().mean())

null rate in the 60s lag columns (should be near 0 -- only symbol-days with <60s of pre-close data would miss):
ref_price_lag60s          0.0
imbalance_ratio_lag60s    0.0
dtype: float64


### Rolling volatility (last 60 seconds)

In [5]:
window = raw.merge(t0, on=["symbol", "date"], how="inner")
window = window[
    (window["ts"] >= (window["t0_ts"] - pd.Timedelta(seconds=60)))
    & (window["ts"] <= window["t0_ts"])  # explicit upper bound: without this, the guarantee that
                                          # nothing after the decision point leaks in relies on
                                          # raw already being truncated before 16:00 and t0 being
                                          # the max ts in that slice -- true today, not enforced
]

roll = window.groupby(["symbol", "date"]).agg(
    ref_price_std_60s=("ref_price", "std"),
    imbalance_ratio_std_60s=("imbalance_ratio", "std"),
    qa_n_snapshots_60s=("ts", "count"),
).reset_index()

panel = panel.merge(roll, on=["symbol", "date"], how="left")
panel["ref_price_std_60s_bps"] = panel["ref_price_std_60s"] / panel["ref_price"] * 1e4

print(panel[["ref_price_std_60s_bps", "imbalance_ratio_std_60s", "qa_n_snapshots_60s"]].describe().T)

                           count       mean       std   min        25%  \
ref_price_std_60s_bps    10430.0   3.973041  5.403060   0.0   1.821346   
imbalance_ratio_std_60s  10430.0   0.018783  0.022752   0.0   0.004164   
qa_n_snapshots_60s       10430.0  60.395686  0.489021  60.0  60.000000   

                               50%        75%         max  
ref_price_std_60s_bps     2.891879   4.541696  283.020441  
imbalance_ratio_std_60s   0.013044   0.024716    0.377123  
qa_n_snapshots_60s       60.000000  61.000000   61.000000  


## Assemble features

In [6]:
drop_cols = [c for c in panel.columns if c.startswith("ref_price_lag") or c.startswith("imbalance_ratio_lag")
             or c.startswith("signed_imbalance_ratio_lag") or c == "ref_price_std_60s"]
features = panel.drop(columns=drop_cols)

print(f"features: {features.shape[0]} rows, {features.shape[1]} columns")
print(f"any nulls in engineered columns: {features.filter(regex='chg_|ret_|std_|urgency|spread|adv|move').isna().sum().sum()}")

features: 10430 rows, 41 columns
any nulls in engineered columns: 32


**41 columns, 10,430 rows, one row per symbol-day — 19 raw/base + 22 engineered.**

- Level (last snapshot): `signed_imbalance_ratio`, `spread_bps`, `urgency` (spread × imbalance interaction),
  `shares_over_adv`, `paired_shares_over_adv`, `near_far_spread_bps` (+ `far_price_available` flag),
  `intraday_move_bps`.
- Time-series (10s/30s/60s lookback, computed from a raw reload of the last 90s of the pre-close window,
  merged back via `merge_asof`): imbalance change, signed-imbalance change, price return in bps.
- Volatility: `ref_price_std_60s_bps`, `imbalance_ratio_std_60s`, `qa_n_snapshots_60s` (coverage check —
  should be ~60-61 for everyone; worth checking before trusting the volatility numbers on any row where it
  isn't. QA-only, not a model input — see the column manifest below).

Target and train/test split follow below. Deliberately not done here, per handling rule 8 in `1) data_exploration_report.md`: no scaling/standardization — any such statistic must be computed only on the training side of the split, which is the modeling notebook's job, not this one's.

## Target

`close_price` (raw level) is **not** a candidate target. It's non-stationary (price levels drift/trend over the month, they don't mean-revert to something a model can learn) and scale-variant (a $2 stock and a $500 stock share no common scale, so a single regression target pooled across the cross-section isn't well-posed -- errors would be dominated by whichever names happen to have the highest price levels, not whichever the model is actually getting wrong). `close_price` stays in the table only as the raw material the three targets below are built from, and for reference.

Three definitions, each a move in bps relative to something observable at the last pre-close snapshot -- stationary and scale-invariant by construction, and directly tied to the take-home's framing (enter after 15:50, exit at 16:00):

- **vs. `ref_price`** -- the price Nasdaq itself would clear at right now, given orders so far.
- **vs. `near_price`** -- Nasdaq's own live estimate of where the *actual* cross will land, folding in the
  book, not just auction orders. Predicting the residual against this is a higher bar: it's only informative
  if there's signal near_price hasn't already captured.
- **vs. book midpoint** -- the simplest, most universal reference, independent of Nasdaq's own auction math.

Confirmed before dividing: `near_price`, `bid`, `ask` are never zero in any of the 10,430 rows, so none of
these need the guard `near_far_spread_bps` needed.

**A caveat on what `target_bps_vs_near` assumes.** Every row here is the 15:59:59 snapshot — this table does not contain features or rows for any earlier decision point, so it can't be used to evaluate an earlier-entry strategy directly. But the target *definition* itself still carries an assumption worth naming: `near_price` isn't disseminated until 15:55 ET, so `target_bps_vs_near` is only a coherent concept for a trader who could actually observe `near_price` before deciding. `target_bps_vs_ref` and `target_bps_vs_mid` don't carry that assumption, since `ref_price`/`bid`/`ask` exist from 15:50 on.

In [7]:
features["target_bps_vs_ref"] = (features["close_price"] / features["ref_price"] - 1) * 1e4
features["target_bps_vs_near"] = (features["close_price"] / features["near_price"] - 1) * 1e4
mid = (features["bid"] + features["ask"]) / 2
features["target_bps_vs_mid"] = (features["close_price"] / mid - 1) * 1e4

# close_price shown for reference only -- not a candidate target, see markdown above
ref_and_targets = ["close_price", "target_bps_vs_ref", "target_bps_vs_near", "target_bps_vs_mid"]
print(features[ref_and_targets].describe().T)
print()
target_cols_only = ["target_bps_vs_ref", "target_bps_vs_near", "target_bps_vs_mid"]
print("nulls:", features[target_cols_only].isna().sum().sum(), "| infs:",
      np.isinf(features[target_cols_only].select_dtypes(include=[np.number])).sum().sum())

                      count        mean         std         min        25%  \
close_price         10430.0  135.615488  279.122285    0.950100  35.442500   
target_bps_vs_ref   10430.0    0.223584    7.287232 -248.756219  -0.800785   
target_bps_vs_near  10430.0   -0.000116    4.343849  -70.546737  -0.478361   
target_bps_vs_mid   10430.0    0.343871    9.782786 -272.952854  -2.248683   

                          50%         75%          max  
close_price         72.240000  148.082500  4946.150000  
target_bps_vs_ref    0.000000    1.404396   239.329876  
target_bps_vs_near   0.000000    0.000000   195.121951  
target_bps_vs_mid    0.458243    2.916090   155.738778  

nulls: 0 | infs: 0


## Column manifest

`target_bps_vs_ref` reconstructs exactly from two plain columns that are both still sitting in this table:
`close_price` and `ref_price`. Same story for the other two targets against `near_price`/the book midpoint.
`close_price` is kept as reference/reconstruction material (not a candidate target -- see above), and `ref_price` is a legitimate feature (it already feeds `spread_bps`, `urgency`, etc.) -- neither can just be dropped from the table. But nothing in a plain CSV says which columns
are which, so the obvious next step in a modeling notebook, `X = train.drop(columns=["target_bps_vs_ref"])`,
would silently leave `close_price` sitting in `X` and produce a perfect, meaningless R².

Three explicit lists below, saved to `data/columns.json` alongside the CSVs so a downstream notebook loads
the split instead of re-deriving it (and re-deriving it wrong).

In [8]:
import json

# close_price is deliberately NOT a candidate target -- non-stationary and scale-variant across
# symbols (see Target section above). Kept as reference/reconstruction material only.
TARGET_COLS = ["target_bps_vs_ref", "target_bps_vs_near", "target_bps_vs_mid"]
IDENTIFIER_COLS = ["symbol", "date", "ts", "local_time", "local_clock", "side",
                   "qa_n_snapshots_60s", "close_price"]
FEATURE_COLS = [c for c in features.columns if c not in TARGET_COLS and c not in IDENTIFIER_COLS]

assert len(TARGET_COLS) + len(IDENTIFIER_COLS) + len(FEATURE_COLS) == features.shape[1], \
    "every column must be classified exactly once"
assert "ref_price" in FEATURE_COLS and "ref_price" not in TARGET_COLS  # feature, not target
assert "close_price" not in TARGET_COLS and "close_price" not in FEATURE_COLS  # reference only, not a target

print(f"{len(TARGET_COLS)} target columns:", TARGET_COLS)
print(f"{len(IDENTIFIER_COLS)} identifier/QA/reference columns:", IDENTIFIER_COLS)
print(f"{len(FEATURE_COLS)} feature columns:", FEATURE_COLS)

with open("data/columns.json", "w") as f:
    json.dump({
        "target_cols": TARGET_COLS,
        "identifier_cols": IDENTIFIER_COLS,
        "feature_cols": FEATURE_COLS,
    }, f, indent=2)
print()
print("wrote data/columns.json")

3 target columns: ['target_bps_vs_ref', 'target_bps_vs_near', 'target_bps_vs_mid']
8 identifier/QA/reference columns: ['symbol', 'date', 'ts', 'local_time', 'local_clock', 'side', 'qa_n_snapshots_60s', 'close_price']
33 feature columns: ['adv', 'ask', 'ask_qty', 'bid', 'bid_qty', 'far_price', 'near_price', 'open', 'paired_shares', 'ref_price', 'shares', 'imbalance_ratio', 'signed_shares', 'signed_imbalance_ratio', 'spread', 'spread_bps', 'urgency', 'shares_over_adv', 'paired_shares_over_adv', 'far_price_available', 'near_far_spread_bps', 'intraday_move_bps', 'imbalance_ratio_chg_10s', 'signed_imbalance_ratio_chg_10s', 'ref_price_ret_10s_bps', 'imbalance_ratio_chg_30s', 'signed_imbalance_ratio_chg_30s', 'ref_price_ret_30s_bps', 'imbalance_ratio_chg_60s', 'signed_imbalance_ratio_chg_60s', 'ref_price_ret_60s_bps', 'imbalance_ratio_std_60s', 'ref_price_std_60s_bps']

wrote data/columns.json


## Train / test split

Split by date, not randomly -- rows on the same day share market-wide conditions, so a random row split
would leak information across train/test. 80/20 by row count, with the boundary snapped to a full trading
day: find the first date where cumulative row count reaches 80% of the total, and put that entire day in
train (never split a day's rows across both sets).

In [9]:
daily_counts = features.groupby("date").size().sort_index()
cum_frac = daily_counts.cumsum() / daily_counts.sum()
split_date = cum_frac[cum_frac >= 0.80].index[0]

train = features[features["date"] <= split_date].copy()
test = features[features["date"] > split_date].copy()

print(f"split date: {split_date.date()} (last day included in train)")
print(f"train: {train.shape[0]} rows ({train.shape[0]/len(features):.1%}), "
      f"{train['date'].nunique()} days, {train['date'].min().date()} to {train['date'].max().date()}")
print(f"test:  {test.shape[0]} rows ({test.shape[0]/len(features):.1%}), "
      f"{test['date'].nunique()} days, {test['date'].min().date()} to {test['date'].max().date()}")
print()
print("no date overlap:", set(train["date"]).isdisjoint(set(test["date"])))

train.to_csv("data/train.csv", index=False)
test.to_csv("data/test.csv", index=False)
print("wrote data/train.csv and data/test.csv")

split date: 2025-03-25 (last day included in train)
train: 8443 rows (80.9%), 17 days, 2025-03-03 to 2025-03-25
test:  1987 rows (19.1%), 4 days, 2025-03-26 to 2025-03-31

no date overlap: True


wrote data/train.csv and data/test.csv


## Feature reference

Predictors only -- the three target columns are documented in the Target section above, not repeated here.

| Feature | Computation | Reason | Citation |
|---|---|---|---|
| `signed_shares` | `shares` if `side=='BUY'`, `-shares` if `side=='SELL'`, `0` if `side=='NONE'` | Direction-aware imbalance size -- raw `shares` alone discards which side is pushing the price | Mechanical encoding, no external source |
| `signed_imbalance_ratio` | `signed_shares / (paired_shares + shares)` | Normalized, direction-aware imbalance; kept at *every* forecast horizon in Lasso feature selection -- the strongest single input found | `research/02_imperial_predicting_us_returns_closing_imbalance.pdf` |
| `spread_bps` | `(ask - bid) / ref_price * 1e4` | Normalized quoted spread -- a $0.05 spread means very different things on a $2 stock vs. a $500 one | `research/01_esann2024_predicting_closing_cross_nasdaq.pdf` (per-stock normalization as an explicit ablation) |
| `urgency` | `signed_imbalance_ratio * spread_bps` | Interaction of imbalance direction/size with available liquidity -- pressure per unit of quoted liquidity | Public Optiver "Trading at the Close" Kaggle kernels (`market_urgency`), see `research/README.md` |
| `shares_over_adv`, `paired_shares_over_adv` | `shares / (adv * 1000)`, same for `paired_shares` | Size normalized by the stock's own trailing volume -- raw share counts span orders of magnitude across symbols | `research/03_arxiv_dynamical_regularities_auctions.pdf` (heavy-tailed/log-normal size distributions) |
| `far_price_available` | `far_price != 0` | Flags the 32 symbol-days where `far_price` never gets a real value (no standalone closing-auction-only orders that day) | `01) data_exploration.ipynb`, Section 3 finding |
| `near_far_spread_bps` | `(near_price / far_price - 1) * 1e4`, `NaN` when `far_price_available` is False | Gap between the two theoretical cross prices -- how much the book and the auction-only orders disagree | `TH _ Nasdaq Closing Auction Prediction.pdf` (brief's own definitions of `near_price`/`far_price`) |
| `intraday_move_bps` | `(ref_price / open - 1) * 1e4` | Full-session move as of the last pre-close snapshot | -- |
| `imbalance_ratio_chg_{10,30,60}s`, `signed_imbalance_ratio_chg_{10,30,60}s` | Current value minus the value `merge_asof`'d from N seconds earlier | Momentum in the imbalance itself, not just its level | `research/02_imperial_predicting_us_returns_closing_imbalance.pdf` (short-horizon imbalance momentum, kept at every horizon alongside the level) |
| `ref_price_ret_{10,30,60}s_bps` | `(ref_price / ref_price_{N}s_ago - 1) * 1e4` | Short-horizon lagged price return | `research/02_imperial_predicting_us_returns_closing_imbalance.pdf`; "lagged features across window periods" pattern in public Optiver kernels |
| `imbalance_ratio_std_60s`, `ref_price_std_60s_bps` | Rolling std over the last 60 seconds of pre-close snapshots | Realized volatility of price/imbalance right before the decision point | `research/03_arxiv_dynamical_regularities_auctions.pdf`, `research/04_arxiv_heavy_tailed_closing_auctions.pdf` |
| `qa_n_snapshots_60s` | Count of raw rows inside the 60s rolling window | QA check, not a model input -- confirms full cadence coverage before trusting the volatility columns on any given row. In `IDENTIFIER_COLS`, not `FEATURE_COLS`, in the column manifest (`data/columns.json`) | -- |

Papers are in `research/` (gitignored, not part of the submission); `research/README.md` has full citations and links.